# High-Bypass Turbofan Flight Profile with Huracan

This notebook uses the open-source **Huracan** Python library to model a 2-spool high-bypass turbofan engine.

We will:
1. Define the engine configuration (Fan, HPC, Combustor, HPT, LPT, Nozzles).
2. Sweep through a small flight profile (takeoff → climb → cruise).
3. Capture **performance metrics** (thrust, fuel flow, SFC/TSFC).
4. Capture **station total temperatures (Tt)** and **pressures (Pt)** after each compressor and turbine.
5. Plot the results.
6. Identify **maximum values** for all temperatures and pressures.


## 1. Imports & Helper Functions

We import Huracan components, atmosphere helpers, and define robust utilities to extract total temperature and pressure at component outlets.

In [1]:
!pip install huracan
from math import pow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from huracan.thermo.fluids import gas, fuel
from huracan.components.channels import inlet, nozzle
from huracan.components.rotary import fan, compressor, turbine
from huracan.components.power.combustion import combustion_chamber
from huracan.engine import shaft, system

def isa_tp(h_m):
    T0 = 288.15; p0 = 101325.0
    L = -0.0065; g0 = 9.80665; R = 287.05
    if h_m <= 11000.0:
        T = T0 + L*h_m
        p = p0 * pow(T/T0, -g0/(L*R))
    else:
        T = 216.65
        p11 = p0 * pow(216.65/T0, -g0/(L*R))
        p = p11 * pow(np.e, -g0*(h_m-11000)/(R*T))
    return T, p

def cp_air(T): return 1004.5
def k_air(T):  return 1.4

def _get_attr(obj, names):
    for n in names:
        if hasattr(obj, n):
            return getattr(obj, n)
    raise AttributeError(f"None of {names} found on {obj}")

def outlet_total_T(comp):
    cand_states = [getattr(comp, n, None) for n in ["state_out", "out", "outlet"]]
    cand_states += [getattr(getattr(comp, "tail", None), "state", None), getattr(comp, "state", None)]
    cand_states = [s for s in cand_states if s is not None]
    s = cand_states[0]
    return _get_attr(s, ["t_0", "T0", "T_0", "tt", "Tt"])

def outlet_total_P(comp):
    cand_states = [getattr(comp, n, None) for n in ["state_out", "out", "outlet"]]
    cand_states += [getattr(getattr(comp, "tail", None), "state", None), getattr(comp, "state", None)]
    cand_states = [s for s in cand_states if s is not None]
    s = cand_states[0]
    return _get_attr(s, ["p_0", "P0", "P_0", "pt", "Pt"])


[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip


## 2. Engine Factory

We define a helper that builds a 2-spool high-bypass turbofan with Fan, HPC, Combustor, HPT, LPT, and nozzles.

The function returns a `system` plus references to each component for probing.

In [2]:
# Build components and a pre-split stream (Inlet → Fan). We will attach gas,
# split, and only THEN wire HPC and run up to it before adding turbine/combustor.

def make_engine_PW4000(
    bpr=5.8,
    pi_fan=1.58, eta_fan=0.91,
    pi_hpc=20.0, eta_hpc=0.89,
    eta_inlet=0.98,
    eta_turb=0.94,
    eta_noz=0.98,
    shaft_eta=0.99,
    comb_eta=0.99, comb_pi=0.97,
    T_t4=1725.0,        # PW4000 typical Tt4 (K)
    LHV=43e6
):
    from huracan.components.channels import inlet, nozzle
    from huracan.components.rotary import fan, compressor, turbine
    from huracan.components.power.combustion import combustion_chamber
    from huracan.thermo.fluids import fuel

    # Components
    In     = inlet(eta=eta_inlet)
    Fan    = fan(eta=eta_fan, PI=pi_fan)
    HPC    = compressor(eta=eta_hpc, PI=pi_hpc)
    Comb   = combustion_chamber(fuel=fuel(LHV=LHV),
                                eta=comb_eta, PI=comb_pi, t01=T_t4)
    HPT    = turbine(eta=eta_turb)
    LPT    = turbine(eta=eta_turb)
    # Mixed nozzle for stability
    NzMix  = nozzle(eta=eta_noz)

    # Pre-split stream: Inlet → Fan
    s_fan = In - Fan

    parts = {
        "Fan": Fan, "HPC": HPC, "Comb": Comb,
        "HPT": HPT, "LPT": LPT, "NzMix": NzMix
    }
    params = {"shaft_eta": shaft_eta, "bpr": bpr}
    return s_fan, parts, params




## 3. Run Flight Profile

This function:
- runs the engine across waypoints,
- logs performance metrics,
- logs Tt/Pt after each major compressor/turbine.

In [3]:
def _prime_until_hpc(s_core, HPC):
    """
    Execute the core stream just far enough that the Fan/HPC have computed
    their required work .w. Different Huracan versions expose different
    run() signatures; try a few in order.
    """
    # Try: positional target
    try:
        s_core.run(HPC)
        return
    except TypeError:
        pass
    # Try: stop=
    try:
        s_core.run(stop=HPC)
        return
    except TypeError:
        pass
    # Try: upto=
    try:
        s_core.run(upto=HPC)
        return
    except TypeError:
        pass
    # If none of the partial-run signatures are supported, do a full run on the core stream.
    # This will compute .w for Fan/HPC; the later eng.run() will still complete fine.
    s_core.run()
# --- add this helper once (e.g., near your other helpers) ---
def _ensure_t0_on(comp):
    """
    Ensure a component has a static temperature attribute 't0' for
    Huracan's stream.v_exit(). Prefer the component's outlet gas state.
    """
    if hasattr(comp, "t0"):
        return
    # Try component outlet state first
    so = getattr(comp, "state_out", None)
    if so is not None and hasattr(so, "t0"):
        comp.t0 = so.t0
        return
    # Fallback: the stream's current gas (post-component)
    st = getattr(comp, "stream", None)
    if st is not None and hasattr(st, "gas") and hasattr(st.gas, "t0"):
        comp.t0 = st.gas.t0
        return
    # Last resort: derive a static guess from total if present (not ideal)
    if so is not None and hasattr(so, "t_0"):
        comp.t0 = float(getattr(so, "t_0"))  # better than None; rarely used

def _try_run_with_params(run_once, profile, base,
                         max_tries=5, margin_K=30.0, verbose=False):
    """
    Run the engine with progressively stronger cycle settings until the
    static temp before nozzle is comfortably above ambient.

    Args:
        run_once: function(profile, pi_fan, pi_hpc, T_t4) -> (perf_df, stn_df)
        profile: list of waypoints
        base: dict with initial 'pi_fan', 'pi_hpc', 'T_t4'
        max_tries: maximum adjustments to try
        margin_K: minimum allowed margin above ambient
        verbose: print tuning steps
    """
    pi_hpc = base.get("pi_hpc", 20.0)
    pi_fan = base.get("pi_fan", 1.60)
    T_t4   = base.get("T_t4",  1700.0)

    last_err = None

    for i in range(max_tries):
        try:
            perf_df, stn_df = run_once(profile, pi_fan, pi_hpc, T_t4)

            # --- margin check ---
            margins = []
            for wp, row in stn_df.iterrows():
                T_amb, _ = isa_tp(row["h_m"])
                # use LPT outlet (just before core nozzle)
                T_pre_noz = row["Tt_lpt"]
                margins.append(T_pre_noz - T_amb)
            min_margin = min(margins)

            if verbose:
                print(f"[check #{i+1}] pi_fan={pi_fan:.2f}, "
                      f"pi_hpc={pi_hpc:.2f}, Tt4={T_t4:.1f}K "
                      f"=> min nozzle margin {min_margin:.1f} K")

            if min_margin >= margin_K:
                return perf_df, stn_df  # ✅ success

            # not enough margin: escalate settings
            if i % 3 == 0:
                pi_hpc *= 1.10        # +10% HPC PR
            elif i % 3 == 1:
                T_t4  += 50.0         # +50 K Tt4
            else:
                pi_fan += 0.05        # +0.05 Fan PR
            continue

        except AssertionError as e:
            last_err = e
            if verbose:
                print(f"[autotune #{i+1}] AssertionError: {e}")
            # escalate in same pattern
            if i % 3 == 0:
                pi_hpc *= 1.10
            elif i % 3 == 1:
                T_t4  += 50.0
            else:
                pi_fan += 0.05
            continue

    if last_err is not None:
        raise last_err
    raise RuntimeError("Autotune exhausted without satisfying margin check")

def run_profile_once(waypoints, mdot_air=450.0,
                     bpr=8.4, pi_fan=1.62, pi_hpc=26.0, T_t4=1800.0,
                     eta_turb=0.95):
    from huracan.thermo.fluids import gas
    from huracan.engine import shaft, system

    perf_rows, station_rows = [], []

    for wp in waypoints:
        h, M = wp["h_m"], wp["M"]
        T_amb, p_amb = isa_tp(h)

        # --- build skeleton ---
        s_fan, parts, params = make_engine_skeleton(
            pi_fan=pi_fan, eta_fan=0.92,
            pi_hpc=pi_hpc, eta_hpc=0.90,
            eta_inlet=0.98,
            eta_turb=eta_turb,
            eta_noz=0.98,
            shaft_eta=0.99,
            comb_eta=0.99, comb_pi=0.97,
            T_t4=T_t4, LHV=43e6
        )

        # unpack references for clarity
        Fan   = parts["Fan"]
        HPC   = parts["HPC"]
        Comb  = parts["Comb"]
        HPT   = parts["HPT"]
        LPT   = parts["LPT"]
        NzCore= parts["NzCore"]
        NzByp = parts["NzByp"]

        # --- attach gas ---
        g = gas(mf=mdot_air, cp=cp_air, k=k_air, m=M, t_0=T_amb, p_0=p_amb)
        g - s_fan
        # Split bypass and core
        core_frac = 1.0 / (1.0 + bpr)
        s_core, s_byp = s_fan.divert(core_frac)
        
        # Core branch: run Fan → HPC first
        s_core - HPC
        s_core.run()
        
        # Hot section
        s_core - Comb - HPT - LPT
        
        # Now mix bypass and core before single nozzle
        s_mixed = s_core + s_byp
        s_mixed - NzMix
        
        # Shafts
        shaft(HPC, HPT, eta=params["shaft_eta"])
        shaft(Fan, LPT, eta=params["shaft_eta"])
        
        # System
        eng = system(s_mixed)
        eng._probe = {"Fan": Fan, "HPC": HPC, "HPT": HPT, "LPT": LPT}
        eng.run(log=False)


        # backfill static temp on LPT (needed for nozzle calc)
        _ensure_t0_on(LPT)

        # performance metrics
        thrust = eng.thrust_flow()
        sfc    = eng.sfc()
        fuel   = eng.fmf()
        tsfc   = (fuel*2.20462*3600) / (thrust*0.224809)

        # nozzle temp margin
        T_pre_noz = outlet_total_T(LPT)
        nozzle_margin = T_pre_noz - T_amb

        perf_rows.append({
            "h_m": h, "M": M,
            "thrust_N": thrust,
            "fuel_kgps": fuel,
            "sfc_kg_per_Ns": sfc,
            "tsfc_lbm_hr_lbf": tsfc,
            "nozzle_margin_K": nozzle_margin
        })

        # station data
        def Tt(c): return outlet_total_T(c)
        def Pt(c): return outlet_total_P(c)
        station_rows.append({
            "h_m": h, "M": M,
            "Tt_fan": Tt(Fan), "Pt_fan": Pt(Fan),
            "Tt_hpc": Tt(HPC), "Pt_hpc": Pt(HPC),
            "Tt_hpt": Tt(HPT), "Pt_hpt": Pt(HPT),
            "Tt_lpt": Tt(LPT), "Pt_lpt": Pt(LPT),
            "nozzle_margin_K": nozzle_margin,
            "T_ambient_K": T_amb
        })

    return pd.DataFrame(perf_rows), pd.DataFrame(station_rows)

def _try_run_with_params(run_once, profile, base,
                         max_tries=5, margin_K=30.0, verbose=False):
    pi_hpc = base.get("pi_hpc", 20.0)
    pi_fan = base.get("pi_fan", 1.60)
    T_t4   = base.get("T_t4",  1700.0)

    last_err = None
    for i in range(max_tries):
        try:
            perf_df, stn_df = run_once(profile, pi_fan, pi_hpc, T_t4)
            min_margin = stn_df["nozzle_margin_K"].min()

            if verbose:
                print(f"[check #{i+1}] pi_fan={pi_fan:.2f}, "
                      f"pi_hpc={pi_hpc:.2f}, Tt4={T_t4:.1f}K "
                      f"=> min nozzle margin {min_margin:.1f} K")

            if min_margin >= margin_K:
                return perf_df, stn_df

            # escalate
            if i % 3 == 0:
                pi_hpc *= 1.10
            elif i % 3 == 1:
                T_t4  += 50.0
            else:
                pi_fan += 0.05
            continue

        except AssertionError as e:
            last_err = e
            if verbose:
                print(f"[autotune #{i+1}] AssertionError: {e}")
            # escalate
            if i % 3 == 0:
                pi_hpc *= 1.10
            elif i % 3 == 1:
                T_t4  += 50.0
            else:
                pi_fan += 0.05
            continue

    if last_err is not None:
        raise last_err
    raise RuntimeError("Autotune exhausted without satisfying margin check")

def run_profile_PW4000(waypoints, mdot_air=450.0):
    """
    Run a flight profile for a Pratt & Whitney PW4000-class turbofan.
    Returns:
        perf_df : DataFrame with thrust, fuel, SFC, TSFC, nozzle margin
        stn_df  : DataFrame with station Tt/Pt and nozzle margin
    """
    from huracan.thermo.fluids import gas
    from huracan.engine import shaft, system

    perf_rows, station_rows = [], []

    # --- PW4000 baseline parameters ---
    bpr     = 5.8
    pi_fan  = 1.58
    pi_hpc  = 20.0
    T_t4    = 1725.0
    eta_turb= 0.94

    # Build skeleton
    s_fan, parts, params = make_engine_PW4000(
        bpr=bpr, pi_fan=pi_fan, pi_hpc=pi_hpc,
        T_t4=T_t4, eta_turb=eta_turb
    )
    Fan, HPC, Comb, HPT, LPT, NzMix = (
        parts["Fan"], parts["HPC"], parts["Comb"],
        parts["HPT"], parts["LPT"], parts["NzMix"]
    )

    for wp in waypoints:
        h, M = wp["h_m"], wp["M"]
        T_amb, p_amb = isa_tp(h)

        # Attach fresh gas for each waypoint
        g = gas(mf=mdot_air, cp=cp_air, k=k_air, m=M, t_0=T_amb, p_0=p_amb)
        g - s_fan

        # Split bypass/core
        core_frac = 1.0 / (1.0 + bpr)
        s_core, s_byp = s_fan.divert(core_frac)

        # Run Fan→HPC first
        s_core - HPC
        s_core.run()


        # Hot section
        s_core - Comb - HPT - LPT - NzCore
        
        # Bypass straight to nozzle
        s_byp - NzByp
        
        # Shafts
        shaft(HPC, HPT, eta=params["shaft_eta"])
        shaft(Fan, LPT, eta=params["shaft_eta"])
        
        # System
        eng = system(s_core, s_byp)
        eng._probe = {"Fan": Fan, "HPC": HPC, "HPT": HPT, "LPT": LPT}
        eng.run(log=False)


        # Ensure LPT has t0 for nozzle calc
        _ensure_t0_on(LPT)

        # Performance
        thrust = eng.thrust_flow()
        sfc    = eng.sfc()
        fuel   = eng.fmf()
        tsfc   = (fuel*2.20462*3600) / (thrust*0.224809)

        T_pre_noz = outlet_total_T(LPT)
        nozzle_margin = T_pre_noz - T_amb

        perf_rows.append({
            "h_m": h, "M": M,
            "thrust_N": thrust,
            "fuel_kgps": fuel,
            "sfc_kg_per_Ns": sfc,
            "tsfc_lbm_hr_lbf": tsfc,
            "nozzle_margin_K": nozzle_margin
        })

        # Station data
        def Tt(c): return outlet_total_T(c)
        def Pt(c): return outlet_total_P(c)
        station_rows.append({
            "h_m": h, "M": M,
            "Tt_fan": Tt(Fan), "Pt_fan": Pt(Fan),
            "Tt_hpc": Tt(HPC), "Pt_hpc": Pt(HPC),
            "Tt_hpt": Tt(HPT), "Pt_hpt": Pt(HPT),
            "Tt_lpt": Tt(LPT), "Pt_lpt": Pt(LPT),
            "nozzle_margin_K": nozzle_margin,
            "T_ambient_K": T_amb
        })

    return pd.DataFrame(perf_rows), pd.DataFrame(station_rows)



## 4. Define Profile & Run

We will test a simple profile: takeoff (0 m, M0.25), climb (3000 m, M0.45), and cruise (9000–11000 m).

In [4]:
profile = [
    {"h_m": 0, "M": 0.25},      # takeoff
    {"h_m": 3000, "M": 0.45},   # climb
    {"h_m": 9000, "M": 0.78},   # high climb
    {"h_m": 11000, "M": 0.80},  # cruise
]

perf_df, stn_df = run_profile_PW4000(profile)

print(perf_df)


0.fn     Fan
          T0 336.514195 [K]
          p0 167207.714 [Pa]
1.m.cp   Compressor
          T0 735.459565 [K]
          p0 2116553.34 [Pa]


NameError: name 'NzCore' is not defined

## 5. Identify Maximum Values

We capture the maximum Tt and Pt values and report the flight conditions at which they occur.

In [ ]:
temp_cols = [c for c in stn_df.columns if c.startswith("Tt_")]
pres_cols = [c for c in stn_df.columns if c.startswith("Pt_")]

maxima = []
for col in temp_cols + pres_cols:
    idx = stn_df[col].idxmax()
    maxima.append({"variable": col,
                   "max_value": stn_df.loc[idx,col],
                   "h_m": stn_df.loc[idx,"h_m"],
                   "M": stn_df.loc[idx,"M"]})
pd.DataFrame(maxima)

## 6. Plot Results

We generate plots for:
- Thrust, Fuel Flow, SFC, TSFC across waypoints
- Tt and Pt after each major component.

In [ ]:
def _axis_labels(df):
    labels = [f"{r.h_m} m, M{r.M}" for r in df.itertuples()]
    return np.arange(len(labels)), labels

x, labels = _axis_labels(perf_df)
plt.plot(x, perf_df["thrust_N"], marker='o'); plt.xticks(x, labels, rotation=30); plt.title("Thrust"); plt.show()
plt.plot(x, perf_df["fuel_kgps"], marker='o'); plt.xticks(x, labels, rotation=30); plt.title("Fuel Flow"); plt.show()
plt.plot(x, perf_df["sfc_kg_per_Ns"], marker='o'); plt.xticks(x, labels, rotation=30); plt.title("SFC"); plt.show()
plt.plot(x, perf_df["tsfc_lbm_hr_lbf"], marker='o'); plt.xticks(x, labels, rotation=30); plt.title("TSFC"); plt.show()

for col in ["Tt_fan","Tt_hpc","Tt_hpt","Tt_lpt"]:
    plt.plot(x, stn_df[col], marker='o'); plt.xticks(x, labels, rotation=30); plt.title(col); plt.show()

for col in ["Pt_fan","Pt_hpc","Pt_hpt","Pt_lpt"]:
    plt.plot(x, stn_df[col], marker='o'); plt.xticks(x, labels, rotation=30); plt.title(col); plt.show()


x, labels = _axis_labels(perf_df)
plt.plot(x, perf_df["nozzle_margin_K"], marker="o")
plt.xticks(x, labels, rotation=30)
plt.ylabel("ΔT [K]")
plt.title("Nozzle Temperature Margin vs Waypoint")
plt.grid(True)
plt.show()
